In [2]:
from time import time

from hetero_isas.monodromy_lp import (
    MonodromyLPDecomposer,
)
from hetero_isas.monodromy_lp.isa import ISAHandler
from numpy.random import Philox
from qiskit.circuit.library import CXGate, CZGate, SwapGate, UnitaryGate, iSwapGate
from qiskit.quantum_info.random import random_unitary
from qiskit.synthesis.two_qubit.xx_decompose import XXDecomposer
from tqdm.notebook import tqdm

generator = Philox(0)

%matplotlib inline

%load_ext snakeviz

In [4]:
isa_handler = ISAHandler(
    [
        CXGate(),
        CXGate().power(1 / 2),
        CXGate().power(1 / 3),
    ],
    [1.0, 1 / 2, 1 / 3],
    ["cx", "sq[2]cx", "sq[3]cx"],
)
mono_lp_decomposer = MonodromyLPDecomposer(isa_handler)

# strengths = XXDecomposer._strength_to_infidelity(1.0)
xx_decomposer = XXDecomposer(1.0)

In [5]:
N = 1_000
qiskit_time = 0
mono_lp_time = 0

for _ in tqdm(range(N)):
    target = random_unitary(4).to_matrix()

    time_start = time()
    xx_decomposer.num_basis_gates(target)
    time_stop = time()
    qiskit_time += time_stop - time_start

    time_start = time()
    mono_lp_decomposer._best_decomposition(target)
    time_stop = time()
    mono_lp_time += time_stop - time_start

print("ratio", mono_lp_time / qiskit_time)
print("avg qiskit", qiskit_time / N)
print("avg monolp", mono_lp_time / N)

  0%|          | 0/1000 [00:00<?, ?it/s]

ratio 3.744186689732629
avg qiskit 0.0012930123805999756
avg monolp 0.0048412797451019285


In [3]:
%reload_ext snakeviz
import cProfile

cProfile.run(
    "mono_lp_decomposer._best_decomposition(target)",
    "../../../docs/profile_timings/temp.prof",
)